# Try Gemma for few-shot simplification

In [1]:
import pathlib as pth

base_location  = pth.Path.cwd().parent.parent

gemma_location  = base_location / "models" / "gemma" / "gemma-2-2b-it-Q6_K.gguf"

## Load Model

I load model through LlamaCpp so it is much lighter

In [2]:
from langchain_community.llms import LlamaCpp

In [3]:
model = LlamaCpp(
            model_path=gemma_location.as_posix(),
            n_gpu_layers=16,
            n_batch=512,
            temperature=0.8,
            max_tokens=256,
            top_p=5,
            verbose=False,
            n_ctx=8192,
            f16_kv=True,
            repeat_penalty=1.1,
        )

I set the temperature lower because we want the model to follow our instructions strictly.

## Let's try the zero-shot simplification tool by levels

In [4]:
def format_prompt(user_text: str, level: int) -> str:
    if level == 1:
        few_shot_examples = (
            """<start_of_turn>user\n"""
            "Simplify the following text in Russian: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Первый фильм будет игровым переложением мультсериала «Бэтмен по ту сторону». Мультсериал начал выходить в прошлом году.<end_of_turn>\n"
            "<start_of_turn>user\n"
            "Simplify the following text in Russian: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Режиссёр Боаз Якин («Свежий») будет ставить фильм.<end_of_turn>\n"
            "<start_of_turn>user\n"
            "Simplify the following text in Russian: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Сейчас уже заключили семь контрактов на выполнение работ в области. Это стоит 3,2 миллиарда рублей.<end_of_turn>\n"
        )

        prompt = (
            """<start_of_turn>user\n"""
            "You are a text simplification tool for the Russian language. "
            "You are given a text in Russian and you need give a simple version of it in Russian. "
            "Your task is to simplify complex sentences into simple ones. "
            "Avoid cramming too many details into one sentence; distribute them across multiple sentences where needed. "
            "Rephrase sentences to remove participial and gerundial constructions. "
            "Where possible, replace passive voice with active voice. "
            "If a sentence consists of only a noun, add a verb. "
            "Replace rare or low-frequency words with more common ones. "
            "Where appropriate, remove or replace foreign words. "
            "Clarify ambiguous phrases by replacing them with more concrete, easily understandable words. "
            "Where possible, avoid words that have paronyms. "
            "Use only Russian, English is forbidden at any cost.\n"
            + few_shot_examples +
            "Simplify the following text in Russian: {user_text}<end_of_turn>\n"
            "<start_of_turn>model\n"
        )

    elif level == 2:
        few_shot_examples = (
            """<start_of_turn>user\n"""
            "Simplify the following text in Russian: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Первый фильм будет версией мультфильма «Бэтмен по ту сторону». Мультфильм вышел в прошлом году.<end_of_turn>\n"
            "<start_of_turn>user\n"
            "Simplify the following text in Russian: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Режиссёр Боаз Якин будет ставить фильм. Он известен по фильму «Свежий».<end_of_turn>\n"
            "<start_of_turn>user\n"
            "Simplify the following text in Russian: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Заключили семь контрактов на выполнение работ. Это стоит 3,2 миллиарда рублей.<end_of_turn>\n"
        )

        prompt = (
            """<start_of_turn>user\n"""
            "You are a text simplification tool for the Russian language. "
            "You are given a text in Russian and you need give a simple version of it in Russian. "
            "Simplify complex or compound sentences by breaking them into shorter ones, aiming for a sentence length of no more than seven words. "
            "Ensure each sentence contains only one idea. "
            "Avoid participial and gerundial constructions, and prefer active voice over passive voice. "
            "Keep essential information like names, nationalities, and roles. Do not remove important details. "
            "Remove unnecessary foreign words (like brand names) and replace rare or long words with simpler, shorter ones. "
            "Simplify ambiguous phrases by using more concrete, clear words. "
            "Remove minor details that do not add significant meaning, but ensure key information remains intact. "
            "Use only Russian, English is forbidden at any cost.\n"
            + few_shot_examples +
            "Simplify the following text in Russian: {user_text}<end_of_turn>\n"
            "<start_of_turn>model\n"
        )

    elif level == 3:
        few_shot_examples = (
            """<start_of_turn>user\n"""
            "Simplify the following text in Russian: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Первый фильм будет снять по мультфильму «Бэтмен по ту сторону». Мультфильм вышел в прошлом году.<end_of_turn>\n"
            "<start_of_turn>user\n"
            "Simplify the following text in Russian: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Фильм снимет режиссёр Боаз Якин. Он известен по фильму «Свежий».<end_of_turn>\n"
            "<start_of_turn>user\n"
            "Simplify the following text in Russian: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Уже заключили договоры на ремонт. Он стоит 3.2 миллиарда рублей.<end_of_turn>\n"
        )

        prompt = (
            """<start_of_turn>user\n"""
            "You are a text simplification assistant for the Russian language. "
            "You are given a text in Russian and you need give a simple version of it in Russian. "
            "Your task is to make the text as simple as possible. "
            "Each sentence should contain only one idea and be no longer than five words. "
            "Remove or replace foreign words (such as names, places, or brands), and avoid minor details. "
            "Eliminate numbers and remove any unnecessary details. "
            "Focus on using the nominative and genitive cases for nouns, and only the present or past tense for verbs. "
            "Avoid passive voice and inverted word order. "
            "Replace rare or low-frequency words with more common ones. "
            "Where possible, replace complex phrases with common expressions, clichés, or idioms. "
            "Remove any extraneous details (if it is possible to without removing original semantics of sentence) and simplify ambiguous phrases as much as possible."
            "Use only Russian, English is forbidden at any cost.\n"
            + few_shot_examples +
            "Simplify the following text in Russian: {user_text}<end_of_turn>\n"
            "<start_of_turn>model\n"
        )

    return prompt.format(user_text=user_text)


### 1 Level

In [5]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=1)

In [6]:
model.invoke(str(prompt))

'Елена Максимова из России стала победительницей конкурса «Миссис Вселенная». \n'

### 2 Level

In [7]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=2)

In [8]:
model.invoke(str(prompt))

'Елена Максимова из России стала победительницей конкурса «Миссис Вселенная». \n\n\n'

### 3 Level

In [9]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=3)

In [10]:
model.invoke(str(prompt))

'Елена Максимова стала Мисс Вселенная. \n'

Well, not bad at all! Let's compute the 200 samples from dataset.

## Load Data


In [11]:
import pandas as pd

data_location = base_location / "data" / "RuSimpleSentAphasia.csv"
data = pd.read_csv(data_location.as_posix())

data.head()

,source,level 1,level 2,level 3
0,Россиянка Елена Максимова одержала победу в ме...,Россиянка Елена Максимова победила в конкурсе ...,Россиянка победила в конкурсе «Миссис Вселенная».,Россиянка победила в конкурсе «Миссис Вселенная».
1,Представительница России впервые завоевала это...,"В прессе сказали, что участница из России полу...",Участница из России получает этот титул впервые.,Россиянка получает этот титул впервые.
2,"Уточняется, что финал прошел в Софии 4 февраля.",Финал прошел в Болгарии в начале февраля.,Финал был в Болгарии в феврале.,Финал был начале февраля. Он был в в Болгарии.
3,Участие в нем принимали 120 женщин из разных с...,В нем участвовали 120 женщин из разных стран.,В нем участвовали 120 женщин из разных стран.,В нем участвовали женщины из разных стран.
4,«Конкуренция на конкурсе была очень жесткая: р...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть.


In [12]:
len(data)

1002

In [13]:
new_data = pd.DataFrame(data["source"].head(200))

In [14]:
new_data

,source
0,Россиянка Елена Максимова одержала победу в ме...
1,Представительница России впервые завоевала это...
2,"Уточняется, что финал прошел в Софии 4 февраля."
3,Участие в нем принимали 120 женщин из разных с...
4,«Конкуренция на конкурсе была очень жесткая: р...
...,...
195,Ранее президент Национального института геофиз...
196,Во время операции врачи извлекли 39 металличес...
197,"По словам одного из хирургов, этот пациент поп..."
198,"Выяснилось, что все эти предметы пациент прогл..."


In [15]:
def apply_generation(text, level) -> str:
    prompt = format_prompt(user_text=text, level=level)
    return model.invoke(str(prompt), stop=["\n\n", " \n\n", ". \n\n"])

In [16]:
for level in [1, 2, 3]:
    column = f"level {level}"
    new_data[column] = new_data["source"].apply(lambda x: apply_generation(x, level))

In [17]:
new_data_location = base_location / "data" / "RuSimpleSentAphasia_200_generated_gemma_few_shot.csv"

new_data.to_csv(new_data_location.as_posix(), index=False)

## Let's calculate BERTscore between ground truth and predicted texts

In [18]:
new_data

,source,level 1,level 2,level 3
0,Россиянка Елена Максимова одержала победу в ме...,Елена Максимова из России завоевала титул Мисс...,Елена Максимова из России выиграла конкурс «Ми...,Елена Максимова стала победительницей конкурса...
1,Представительница России впервые завоевала это...,"Российская делегация впервые won this title, a...",Россиянка впервые стала победителем. Такое со...,Россиянка впервые выиграла конкурс. \n
2,"Уточняется, что финал прошел в Софии 4 февраля.",Финал состоялся 4 февраля в Софии. \n,Финал прошёл в Софии 4 февраля. \n,Финал прошёл в Софии 4 февраля. \n
3,Участие в нем принимали 120 женщин из разных с...,В нем приняли участие 120 женщин из разных стр...,"Женщины из 12 стран участвовали. Это Китай, Та...",В нём участвовали 120 женщин из разных стран. \n
4,«Конкуренция на конкурсе была очень жесткая: р...,Конкурс был очень конкурентным: много участник...,Конкурс был очень конкурентоспособным. Много у...,Конкурс был очень конкурентным. \n
...,...,...,...,...
195,Ранее президент Национального института геофиз...,Президент INGV Италии (Карло Дольони) рассказа...,Землетрясение в Турции сместило литосферные плиты,Землетрясение в Турции вызвало сдвиг плит
196,Во время операции врачи извлекли 39 металличес...,Во время операции хирурги извлечили 39 предмет...,В ходе операции из живота 30-летнего ливанца б...,В ходе операции из живота 30-летнего мужчины и...
197,"По словам одного из хирургов, этот пациент поп...","Хирург рассказал о пациенте, который попал в б...",Хирург рассказал о пациенте с приступом удушья...,"Хирург сказал, что пациент захлебнулся шлангом"
198,"Выяснилось, что все эти предметы пациент прогл...",Пациент проглотил все эти предметы за год,Пациент проглотил все предметы за год. \n,Пациент проглотил всё за год


In [19]:
from evaluate import load
import numpy as np

bertscore = load("bertscore")

bert_scores = {}
val_data = data.head(200)


for level in val_data.columns[1:]:
    references = val_data[level].tolist()
    predictions = new_data[level].tolist()

    results = bertscore.compute(predictions=predictions, references=references, lang="ru")
    
    bert_scores[level] = {
        "precision": np.mean(results['precision']),
        "recall": np.mean(results['recall']),
        "f1": np.mean(results['f1'])
    }

/home/z00logist/hse-simplification-for-aphasia/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/z00logist/hse-simplification-for-aphasia/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [20]:
for level, scores in bert_scores.items():
    print(f"BERTScore for {level}:")
    print(f"Precision: {scores['precision']}")
    print(f"Recall: {scores['recall']}")
    print(f"F1: {scores['f1']}\n")

BERTScore for level 1:
Precision: 0.7882860207557678
Recall: 0.8131936809420586
F1: 0.7998759236931801

BERTScore for level 2:
Precision: 0.7647380986809731
Recall: 0.7798201751708984
F1: 0.7714425811171531

BERTScore for level 3:
Precision: 0.7590590259432792
Recall: 0.7633264455199241
F1: 0.7604610559344291



For gemma the few-shot method was rather unsuccessful.